<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/ml-fundamentals/classification-metrics/classification-metrics.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("classification-metrics", ...)`


# Classification Metrics from Scratch

**Difficulty**: 🟡 Medium

### Problem Statement

Evaluation code is where silent bugs live. A transposed confusion matrix, a "macro-F1" that is really a harmonic mean, an AUC that depends on how ties happen to be ordered, an "average precision" computed as a trapezoid, a calibration error that drops every 100%-confident prediction: each of these produces a plausible number that is wrong. Interviewers for applied-ML and research roles like these metrics because getting them exactly right means knowing what they measure.

Implement, in vectorized PyTorch, the metrics behind `sklearn.metrics.confusion_matrix`, `precision_recall_fscore_support`, `roc_auc_score`, `average_precision_score`, and the Expected Calibration Error of Guo et al., *On Calibration of Modern Neural Networks* (ICML 2017).

---

### Requirements

1. **`confusion_matrix(y_true, y_pred, num_classes) -> LongTensor`** of shape `(C, C)`: entry `[i, j]` counts samples whose **true** class is `i` and **predicted** class is `j` (rows = truth, columns = prediction). No Python loop over samples — a single counting op is enough.

2. **`precision_recall_f1(y_true, y_pred, num_classes, average) -> (precision, recall, f1)`** for single-label multiclass predictions, with **0/0 := 0** everywhere:
   - per class: $P_c = \frac{TP_c}{TP_c + FP_c}$, $R_c = \frac{TP_c}{TP_c + FN_c}$, $F1_c = \frac{2 P_c R_c}{P_c + R_c} = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c}$;
   - `average=None` → three `(C,)` float tensors (classes that never occur get 0);
   - `average="macro"` → the **unweighted mean of the per-class values** over the classes that occur in `y_true` or `y_pred` ($TP_c + FP_c + FN_c > 0$; what sklearn does by default and torchmetrics does for multiclass). Macro-F1 is the mean of per-class F1, **not** the harmonic mean of macro-P and macro-R;
   - `average="micro"` → pool $TP$, $FP$, $FN$ over classes, then apply the formulas once.

   Scalars may be returned as 0-dim tensors or Python floats.

3. **`roc_auc(scores, labels) -> float`** — binary. `labels` holds 0/1 (any integer or bool dtype) and `scores` are real numbers (higher = more positive). ROC-AUC is the probability that a random positive outranks a random negative, with **ties counting ½**:
   $$\mathrm{AUC} = \frac{1}{n_+ n_-}\sum_{i \in +}\sum_{j \in -}\Big(\mathbb{1}[s_i > s_j] + \tfrac12 \mathbb{1}[s_i = s_j]\Big)$$
   Compute it in **$O(n \log n)$** with ranks (the Mann–Whitney $U$ statistic): give tied scores their **average rank**, then $\mathrm{AUC} = \big(\sum_{i \in +} \mathrm{rank}_i - \tfrac{n_+(n_+ + 1)}{2}\big) / (n_+ n_-)$. Never build the $n_+ \times n_-$ comparison matrix. You may assume both classes are present.

4. **`average_precision(scores, labels) -> float`** — sklearn's step-wise definition. With the distinct score values sorted in decreasing order $t_1 > t_2 > \dots$, and $P_n, R_n$ the precision and recall of "predict positive iff score $\ge t_n$":
   $$\mathrm{AP} = \sum_n (R_n - R_{n-1})\, P_n, \qquad R_0 = 0.$$
   Tied scores form **one** threshold. There is no interpolation and no trapezoid. $O(n \log n)$. You may assume at least one positive.

5. **`expected_calibration_error(probs, labels, n_bins=15) -> float`** — `probs` is `(N, C)` with rows summing to 1, `labels` is `(N,)`. The confidence is $\hat p_i = \max_c \texttt{probs}[i, c]$ and the prediction is the argmax. Use `n_bins` **equal-width, right-closed** bins $B_b = \{i : \hat p_i \in (\tfrac{b}{n}, \tfrac{b+1}{n}]\}$, so a confidence of exactly 1.0 lands in the last bin:
   $$\mathrm{ECE} = \sum_{b} \frac{|B_b|}{N}\,\big|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\big|,$$
   where $\mathrm{acc}$ and $\mathrm{conf}$ are the mean correctness and mean confidence inside a bin. Empty bins contribute 0.

---

### Constraints

- ✅ Pure PyTorch, vectorized: `bincount`, `sort`, `cumsum`, `unique_consecutive`, `bucketize`, `scatter_add` are your friends.
- ❌ No `sklearn`, `torchmetrics` or `scipy` in your implementation (`sklearn` is used only in the optional cross-check cell).
- ❌ No Python loop over samples, and no $O(n^2)$ pairwise or all-thresholds matrices for AUC/AP.
- ✅ A loop over the `n_bins` bins in ECE is acceptable (the reference code of Guo et al. does that), though a `bincount` version is neater.

---

<details>
  <summary>💡 Hint</summary>

  - How many distinct `(true, pred)` pairs are there? Can you give each one a unique integer id?
  - Once you have the confusion matrix, where do TP, FP and FN live — the diagonal, the rows, the columns?
  - For AUC: if you sort all scores ascending, what does the rank of a positive sample tell you about how many negatives it beats? What should tied samples receive so that each positive–negative tie counts exactly ½?
  - For AP: after sorting by score (descending), a cumulative sum of the labels gives the TP count at every cut-off, but you may only cut *between* different score values.
  - ECE can be written as $\frac1N \sum_b \big|\sum_{i \in B_b} (\text{correct}_i - \hat p_i)\big|$.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)
num_classes, N = 4, 2000

# A synthetic 4-class classifier that is over-confident: informative logits sharpened by T = 0.5.
y_true = torch.randint(0, num_classes, (N,))
logits = torch.randn(N, num_classes) + 1.5 * F.one_hot(y_true, num_classes)
probs = torch.softmax(logits / 0.5, dim=1)
y_pred = probs.argmax(dim=1)

# A binary problem with ~10% positives and heavily tied (rounded) scores.
y_bin = (torch.rand(N) < 0.1).long()
s_bin = ((torch.randn(N) + 1.2 * y_bin) * 4).round() / 4
print(f"accuracy {(y_true == y_pred).float().mean():.3f}, positives {int(y_bin.sum())}/{N}, "
      f"distinct binary scores {s_bin.unique().numel()}")

In [ ]:
def confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int) -> torch.Tensor:
    """
    Args:
        y_true: (N,) int64 true classes in [0, num_classes).
        y_pred: (N,) int64 predicted classes in [0, num_classes).
        num_classes: C.

    Returns:
        (C, C) int64 tensor; [i, j] = #samples with true class i predicted as j.
    """
    # TODO: one unique id per (true, pred) pair, count them all at once, reshape to (C, C)
    raise NotImplementedError

In [ ]:
def confusion_loop(y, p, C):
    cm = torch.zeros(C, C, dtype=torch.long)
    for t, q in zip(y.tolist(), p.tolist()):
        cm[t, q] += 1
    return cm

cm = confusion_matrix(y_true, y_pred, num_classes)
print(cm)
assert cm.dtype == torch.long and cm.shape == (num_classes, num_classes)
assert torch.equal(cm, confusion_loop(y_true, y_pred, num_classes))
assert confusion_matrix(torch.tensor([0, 0, 1]), torch.tensor([1, 1, 1]), 2).tolist() == [[0, 2], [0, 1]], \
    "rows must be the TRUE class"
print("confusion_matrix tests passed")

In [ ]:
def precision_recall_f1(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int,
                        average="macro"):
    """
    Args:
        y_true, y_pred: (N,) int64 class ids.
        num_classes: C.
        average: None (per class), "macro" or "micro".

    Returns:
        (precision, recall, f1): (C,) tensors for average=None, scalars otherwise. 0/0 := 0.
    """
    # TODO: TP, FP, FN per class from the confusion matrix
    # TODO: micro -> pool counts first; macro -> mean over classes with TP + FP + FN > 0
    raise NotImplementedError

In [ ]:
P, R, F1 = precision_recall_f1(y_true, y_pred, num_classes, None)
print("per-class P", P, "\nper-class R", R, "\nper-class F1", F1)
acc = (y_true == y_pred).double().mean()
mP, mR, mF = precision_recall_f1(y_true, y_pred, num_classes, "micro")
assert torch.allclose(torch.stack([torch.as_tensor(v).double() for v in (mP, mR, mF)]), acc.expand(3)), \
    "single-label multiclass: micro P = R = F1 = accuracy"
MP, MR, MF = precision_recall_f1(y_true, y_pred, num_classes, "macro")
assert abs(float(MF) - float(F1.mean())) < 1e-9, "macro-F1 is the mean of the per-class F1 scores"

# A class that is never predicted: precision 0/0 -> 0, no NaN
y = torch.tensor([0, 1, 2, 2]); p = torch.tensor([0, 1, 1, 1])
P, R, F1 = precision_recall_f1(y, p, 3, None)
assert not torch.isnan(P).any() and float(P[2]) == 0.0 and float(R[2]) == 0.0
print("precision_recall_f1 tests passed")

In [ ]:
def roc_auc(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """
    Binary ROC-AUC = P(score_pos > score_neg) + 0.5 * P(tie), in O(n log n).

    Args:
        scores: (N,) real-valued scores.
        labels: (N,) 0/1 labels (int or bool).

    Returns:
        AUC as a Python float.
    """
    # TODO: sort scores; find runs of equal scores; give each run its average (1-based) rank
    # TODO: U = sum of positive ranks - n_pos (n_pos + 1) / 2;  AUC = U / (n_pos * n_neg)
    raise NotImplementedError

In [ ]:
def auc_pairwise(s, y):
    # O(n^2) definition, fine for a test
    s, y = s.double(), y.bool()
    d = s[y][:, None] - s[~y][None, :]
    return float((d > 0).double().mean() + 0.5 * (d == 0).double().mean())

auc = roc_auc(s_bin, y_bin)
print(f"AUC = {auc:.6f}")
assert abs(auc - auc_pairwise(s_bin, y_bin)) < 1e-9, "ties must count 1/2"
assert abs(roc_auc(s_bin, y_bin.bool()) - auc) < 1e-12, "bool labels work too"
yy = torch.tensor([0, 0, 1, 1]); ss = torch.tensor([0.1, 0.4, 0.35, 0.8])
assert abs(roc_auc(ss, yy) - 0.75) < 1e-9
assert abs(roc_auc(-ss, yy) - 0.25) < 1e-9
assert abs(roc_auc(torch.ones(4), yy) - 0.5) < 1e-9, "constant scores -> 0.5"
print("roc_auc tests passed")

In [ ]:
def average_precision(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """
    sklearn-style AP = sum_n (R_n - R_{n-1}) * P_n over distinct score thresholds.

    Args:
        scores: (N,) real-valued scores.
        labels: (N,) 0/1 labels (int or bool).

    Returns:
        AP as a Python float.
    """
    # TODO: sort descending; cumulative TP; evaluate only at the LAST index of each tie run
    # TODO: precision = TP / #predicted, recall = TP / #positives; sum recall steps * precision
    raise NotImplementedError

In [ ]:
def ap_by_thresholding(s, y):
    # Brute force: threshold at every distinct score, highest first.
    s, y = s.double(), y.double()
    ap, r_prev = 0.0, 0.0
    for t in s.unique().flip(0):
        sel = s >= t
        tp = float(y[sel].sum())
        ap += (tp / float(y.sum()) - r_prev) * tp / float(sel.sum())
        r_prev = tp / float(y.sum())
    return ap

ap = average_precision(s_bin, y_bin)
print(f"AP = {ap:.6f}   (positive rate {y_bin.double().mean():.3f})")
assert abs(ap - ap_by_thresholding(s_bin, y_bin)) < 1e-9
assert abs(average_precision(torch.tensor([.9, .8, .7, .6]), torch.tensor([1, 0, 1, 0])) - (0.5 + 0.5 * 2 / 3)) < 1e-9
assert abs(average_precision(torch.full((10,), 0.3), torch.tensor([1, 0, 0, 1, 0, 0, 0, 0, 0, 0])) - 0.2) < 1e-9, \
    "all scores tied -> one threshold -> AP = positive rate"
print("average_precision tests passed")

In [ ]:
def expected_calibration_error(probs: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    """
    Top-label ECE with equal-width, right-closed confidence bins (b/n, (b+1)/n].

    Args:
        probs: (N, C) predicted class probabilities (rows sum to 1).
        labels: (N,) int64 true classes.
        n_bins: number of bins.

    Returns:
        ECE as a Python float.
    """
    # TODO: confidence = max prob, prediction = argmax, correct = (pred == label)
    # TODO: bin index of each confidence (right-closed bins)
    # TODO: sum over bins of |B|/N * |acc(B) - conf(B)|
    raise NotImplementedError

In [ ]:
def ece_loop(probs, labels, n_bins=15):
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).double()
    ece = 0.0
    for b in range(n_bins):
        in_bin = (conf > b / n_bins) & (conf <= (b + 1) / n_bins)
        if in_bin.any():
            ece += in_bin.double().mean() * (correct[in_bin].mean() - conf[in_bin].double().mean()).abs()
    return float(ece)

ece = expected_calibration_error(probs, y_true)
print(f"ECE (T = 0.5, over-confident) = {ece:.4f}")
assert abs(ece - ece_loop(probs, y_true)) < 1e-6
# 75%-confident and right 3 times out of 4, plus certain and right: perfectly calibrated
calibrated = torch.tensor([[0.75, 0.25]] * 4 + [[0.0, 1.0]] * 2)
assert abs(expected_calibration_error(calibrated, torch.tensor([0, 0, 0, 1, 1, 1]))) < 1e-9
# confidence exactly 1.0 must be counted (last bin is right-closed)
assert abs(expected_calibration_error(torch.tensor([[1.0, 0.0]]), torch.tensor([1])) - 1.0) < 1e-9
print("expected_calibration_error tests passed")

In [ ]:
# Optional: cross-check against scikit-learn if it is installed (it is on Colab).
try:
    from sklearn import metrics as skm
except ImportError:
    print("scikit-learn not installed -- skipping the cross-check")
else:
    assert (confusion_matrix(y_true, y_pred, num_classes).numpy()
            == skm.confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))).all()
    for avg in ("macro", "micro"):
        ours = [float(v) for v in precision_recall_f1(y_true, y_pred, num_classes, avg)]
        theirs = skm.precision_recall_fscore_support(y_true, y_pred, average=avg, zero_division=0)[:3]
        assert max(abs(a - b) for a, b in zip(ours, theirs)) < 1e-9, (avg, ours, theirs)
    assert abs(roc_auc(s_bin, y_bin) - skm.roc_auc_score(y_bin, s_bin)) < 1e-9
    assert abs(average_precision(s_bin, y_bin) - skm.average_precision_score(y_bin, s_bin)) < 1e-9
    print("matches sklearn.metrics")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("classification-metrics", confusion_matrix=confusion_matrix,
      precision_recall_f1=precision_recall_f1, roc_auc=roc_auc,
      average_precision=average_precision, expected_calibration_error=expected_calibration_error)